In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings , ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

# step 1 : loading pdf / loder
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs=loader.load()
# print(len(docs))

# step 2 : splitting in chunks / spliter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data=splitter.split_documents()
# len(splitted_data)

# step 3 : emmbeddings data

embeddings =OpenAIEmbeddings(model="text-embedding-3-large")

# step 4 : storing in vector db
vector_store=Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

query = "Machine Learings and Data Science content"

data=vector_store.similarity_search(query=query)

# len(data)

context=""

for doc in data:
    context+=doc.page_content + "\n"

llm=ChatOpenAI("gpt-5")

res =llm.invoke(f"can you provide me the answer based on provided context for my question , context={context} and question: {query}")

# chain => conetext_generator | prompt | llm | str_parser

def get_context(query:str):
    data=vector_store.similarity_search(query=query)
    context=""

    for doc in data:
     context+=doc.page_content + "\n"
    return {
        "context":context,
        "question":query
    }


prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

rag_chain = get_context | prompt | llm
res = rag_chain.invoke("What is the value of RBC Count and is it in range ? ")

print(res.content)